# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a tutorial for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will walk through loading the metadata, inspecting the available record sets and fields (referencing them by their `@id`), extracting tabular data, performing exploratory data analysis, and visualizing results.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s. We will reference all entities by their `@id` as required for reproducibility.

Let's enumerate the Record Sets, their fields, and columns, using only their Croissant `@id`s.

In [ ]:
# List record sets with @id
print("Available Record Sets (by @id):")
record_sets = [r['@id'] for r in metadata.to_json().get('recordSet', [])]
for rs in record_sets:
    print(f"  - {rs}")

# List fields and columns for each record set
for record_set_id in record_sets:
    print(f"\nRecord Set @id: {record_set_id}")
    rs = metadata.get_by_id(record_set_id)
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"Fields (@id):")
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', field)
        else:
            field_id = field
        print(f"  - {field_id}")

    # Now list the columns for each field
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', field)
        else:
            field_id = field
        field_obj = metadata.get_by_id(field_id)
        columns = field_obj.get('column', [])
        if columns:
            if isinstance(columns, dict):
                columns = [columns]
            print(f"    Columns of {field_id} (@id):")
            for col in columns:
                if isinstance(col, dict):
                    col_id = col.get('@id', col)
                else:
                    col_id = col
                print(f"      - {col_id}")

## 3. Data Extraction

Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview. 

We'll demonstrate by extracting the primary protein table, referencing everything by `@id`.

In [ ]:
# Define the record set @id to extract (update as needed after viewing the previous cell's output)
# Suppose, based on output, the main protein record set is '@id': 'https://api.app.sen.science/frontiers/7154140/recordsets/protein_table'
# For demonstration, we'll call this variable 'protein_table_id'
protein_table_id = 'https://api.app.sen.science/frontiers/7154140/recordsets/protein_table'
# You may need to set this string to the actual @id if different in your listing

# List all available record sets here
record_sets = [protein_table_id]

dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

df = dataframes[protein_table_id]

print("Columns present in the protein record set:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

We will perform common data processing steps: filtering by a numeric field (referenced by its `@id`), normalizing, and grouping by a categorical field (`@id`).

First, we identify a numeric column such as 'mw' (molecular weight, or use its Croissant `@id`/column `@id`), and a group field such as 'sample_id' (update the names to match their actual column names in the DataFrame if different).

In [ ]:
# Select a numeric field and a group field for analysis
# Replace with the actual column name or Croissant field/column @id from df.columns if needed
numeric_field = 'mw'  # e.g., molecular weight
group_field = 'sample_id'  # e.g., the sample the protein was detected in

if numeric_field in df.columns:
    threshold = 10000  # Example threshold for mw
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Column '{numeric_field}' not found. Available columns: {df.columns.tolist()}")

## 5. Visualization

Visualize the distribution of the molecular weight, and its grouped means if available.

We use matplotlib to plot histograms and bar charts. Update the column names as needed if different in your DataFrame.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the molecular weight field (mw)
if numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=40)
    plt.title('Distribution of Molecular Weight')
    plt.xlabel('Molecular Weight')
    plt.ylabel('Frequency')
    plt.show()

if 'grouped_df' in locals():
    # Bar chart of group means
    plt.figure(figsize=(10,4))
    plt.bar(grouped_df[group_field], grouped_df[numeric_field])
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset package via the `mlcroissant` library, referencing all fields, record sets, and columns by their `@id` per Croissant best practices. We inspected available tables and fields, extracted tabular data, performed basic analysis (including filtering and normalization), and visualized distributions and grouped metrics. This dataset supports a wide variety of analyses including protein expression, comparative studies, and biomarker discovery.

You are encouraged to further explore the available record sets, fields, and visualizations by adapting the code to reference additional `@id`s as needed for your specific research questions!